In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# CONFIGURATION
# ============================================================

BRONZE = "ecommerce_returns.bronze"
SILVER = "ecommerce_returns.silver"
QUARANTINE = "ecommerce_returns.quarantine"


# ============================================================
# HELPER FUNCTION
# ============================================================

def write_quarantine(df, table_name):
    """
    Writes rejected records into a quarantine Delta table.
    """
    if df.limit(1).count() > 0:
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(f"{QUARANTINE}.{table_name}")
        )
        print(f"Quarantine table created: {QUARANTINE}.{table_name}")
    else:
        print(f"No rejected records for: {table_name}")


# ============================================================
# 1. CUSTOMERS
# ============================================================

customers = spark.table(f"{BRONZE}.customers")

customers_clean = (
    customers
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("customer_name", F.trim(F.col("customer_name")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("customer_segment", F.trim(F.col("customer_segment")))
)

# Quarantine records with missing customer_id
customers_bad = (
    customers_clean
    .filter(
        F.col("customer_id").isNull() |
        (F.col("customer_id") == "")
    )
    .withColumn("quarantine_reason", F.lit("Missing customer_id"))
)

write_quarantine(customers_bad, "customers_rejected")

# Keep valid customers and remove duplicates
customers_silver = (
    customers_clean
    .filter(
        F.col("customer_id").isNotNull() &
        (F.col("customer_id") != "")
    )
    .dropDuplicates(["customer_id"])
)

(
    customers_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER}.customers")
)

print("Created Silver table: customers")


# ============================================================
# 2. SELLERS
# ============================================================

sellers = spark.table(f"{BRONZE}.sellers")

sellers_clean = (
    sellers
    .withColumn("seller_id", F.trim(F.col("seller_id")))
    .withColumn("seller_name", F.trim(F.col("seller_name")))
    .withColumn("seller_region", F.trim(F.col("seller_region")))
    .withColumn("seller_rating", F.col("seller_rating").cast("double"))
    .withColumn("seller_join_date", F.to_date("seller_join_date"))
)

sellers_bad = (
    sellers_clean
    .filter(
        F.col("seller_id").isNull() |
        (F.col("seller_id") == "")
    )
    .withColumn("quarantine_reason", F.lit("Missing seller_id"))
)

write_quarantine(sellers_bad, "sellers_rejected")

sellers_silver = (
    sellers_clean
    .filter(
        F.col("seller_id").isNotNull() &
        (F.col("seller_id") != "")
    )
    .dropDuplicates(["seller_id"])
)

(
    sellers_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER}.sellers")
)

print("Created Silver table: sellers")


# ============================================================
# 3. PRODUCTS
# ============================================================

products = spark.table(f"{BRONZE}.products")

products_clean = (
    products
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn("subcategory", F.trim(F.col("subcategory")))
    .withColumn("brand", F.trim(F.col("brand")))
    .withColumn("seller_id", F.trim(F.col("seller_id")))
    .withColumn("cost_price", F.col("cost_price").cast("double"))
    .withColumn("selling_price", F.col("selling_price").cast("double"))
    .withColumn("weight_grams", F.col("weight_grams").cast("double"))
)

products_bad = (
    products_clean
    .filter(
        F.col("product_id").isNull() |
        (F.col("product_id") == "")
    )
    .withColumn("quarantine_reason", F.lit("Missing product_id"))
)

write_quarantine(products_bad, "products_rejected")

products_silver = (
    products_clean
    .filter(
        F.col("product_id").isNotNull() &
        (F.col("product_id") != "")
    )
    .dropDuplicates(["product_id"])
)

(
    products_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER}.products")
)

print("Created Silver table: products")


# ============================================================
# 4. ORDERS
# ============================================================

orders = spark.table(f"{BRONZE}.orders")

orders_clean = (
    orders
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("seller_id", F.trim(F.col("seller_id")))
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("discount", F.col("discount").cast("double"))
    .withColumn("payment_method", F.trim(F.col("payment_method")))
    .withColumn("order_status", F.trim(F.col("order_status")))
)


# ------------------------------------------------------------
# Find invalid orders
# ------------------------------------------------------------

orders_invalid = (
    orders_clean
    .filter(
        F.col("order_id").isNull() |
        (F.col("order_id") == "") |
        F.col("customer_id").isNull() |
        (F.col("customer_id") == "") |
        F.col("quantity").isNull() |
        (F.col("quantity") <= 0) |
        F.col("order_date").isNull()
    )
    .withColumn(
        "quarantine_reason",
        F.concat_ws(
            "; ",
            F.when(
                F.col("order_id").isNull() |
                (F.col("order_id") == ""),
                F.lit("Missing order_id")
            ),
            F.when(
                F.col("customer_id").isNull() |
                (F.col("customer_id") == ""),
                F.lit("Missing customer_id")
            ),
            F.when(
                F.col("quantity").isNull() |
                (F.col("quantity") <= 0),
                F.lit("Invalid quantity")
            ),
            F.when(
                F.col("order_date").isNull(),
                F.lit("Invalid order_date")
            )
        )
    )
)

write_quarantine(orders_invalid, "orders_rejected")


# ------------------------------------------------------------
# Keep valid orders
# ------------------------------------------------------------

orders_valid = (
    orders_clean
    .filter(
        F.col("order_id").isNotNull() &
        (F.col("order_id") != "") &
        F.col("customer_id").isNotNull() &
        (F.col("customer_id") != "") &
        F.col("quantity").isNotNull() &
        (F.col("quantity") > 0) &
        F.col("order_date").isNotNull()
    )
)


# ------------------------------------------------------------
# Detect duplicate orders
# ------------------------------------------------------------

order_window = Window.partitionBy("order_id").orderBy(
    F.col("order_date").asc()
)

orders_ranked = (
    orders_valid
    .withColumn("duplicate_rank", F.row_number().over(order_window))
)

orders_duplicates = (
    orders_ranked
    .filter(F.col("duplicate_rank") > 1)
    .drop("duplicate_rank")
    .withColumn(
        "quarantine_reason",
        F.lit("Duplicate order_id")
    )
)

write_quarantine(orders_duplicates, "orders_duplicates")


# ------------------------------------------------------------
# Final clean orders
# ------------------------------------------------------------

orders_silver = (
    orders_ranked
    .filter(F.col("duplicate_rank") == 1)
    .drop("duplicate_rank")
)

(
    orders_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER}.orders")
)

print("Created Silver table: orders")


# ============================================================
# 5. RETURNS
# ============================================================

returns = spark.table(f"{BRONZE}.returns")

returns_clean = (
    returns
    .withColumn("return_id", F.trim(F.col("return_id")))
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("return_date", F.to_date("return_date"))
    .withColumn(
        "return_reason",
        F.initcap(F.trim(F.col("return_reason")))
    )
    .withColumn("return_quantity", F.col("return_quantity").cast("int"))
    .withColumn("return_status", F.trim(F.col("return_status")))
    .withColumn("condition", F.trim(F.col("condition")))
)


# ------------------------------------------------------------
# Join to orders for validation
# ------------------------------------------------------------

returns_with_orders = returns_clean.join(
    orders_silver.select(
        "order_id",
        "order_date"
    ),
    on="order_id",
    how="left"
)


# ------------------------------------------------------------
# Invalid returns
# ------------------------------------------------------------

returns_invalid = (
    returns_with_orders
    .filter(
        F.col("return_id").isNull() |
        (F.col("return_id") == "") |
        F.col("order_date").isNull() |
        F.col("return_date").isNull() |
        (F.col("return_date") < F.col("order_date")) |
        F.col("return_quantity").isNull() |
        (F.col("return_quantity") <= 0)
    )
    .withColumn(
        "quarantine_reason",
        F.concat_ws(
            "; ",
            F.when(
                F.col("return_id").isNull() |
                (F.col("return_id") == ""),
                F.lit("Missing return_id")
            ),
            F.when(
                F.col("order_date").isNull(),
                F.lit("Unknown order_id")
            ),
            F.when(
                F.col("return_date").isNull(),
                F.lit("Invalid return_date")
            ),
            F.when(
                F.col("return_date") < F.col("order_date"),
                F.lit("Return date before order date")
            ),
            F.when(
                F.col("return_quantity").isNull() |
                (F.col("return_quantity") <= 0),
                F.lit("Invalid return quantity")
            )
        )
    )
)

write_quarantine(returns_invalid, "returns_rejected")


# ------------------------------------------------------------
# Valid returns
# ------------------------------------------------------------

returns_valid = (
    returns_with_orders
    .filter(
        F.col("return_id").isNotNull() &
        (F.col("return_id") != "") &
        F.col("order_date").isNotNull() &
        F.col("return_date").isNotNull() &
        (F.col("return_date") >= F.col("order_date")) &
        F.col("return_quantity").isNotNull() &
        (F.col("return_quantity") > 0)
    )
)


# ------------------------------------------------------------
# Remove duplicate returns
# ------------------------------------------------------------

return_window = Window.partitionBy("return_id").orderBy(
    F.col("return_date").asc()
)

returns_ranked = (
    returns_valid
    .withColumn(
        "duplicate_rank",
        F.row_number().over(return_window)
    )
)

returns_duplicates = (
    returns_ranked
    .filter(F.col("duplicate_rank") > 1)
    .drop("duplicate_rank")
    .withColumn(
        "quarantine_reason",
        F.lit("Duplicate return_id")
    )
)

write_quarantine(returns_duplicates, "returns_duplicates")


returns_silver = (
    returns_ranked
    .filter(F.col("duplicate_rank") == 1)
    .drop("duplicate_rank", "order_date")
)

(
    returns_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER}.returns")
)

print("Created Silver table: returns")


# ============================================================
# 6. REFUNDS
# ============================================================

refunds = spark.table(f"{BRONZE}.refunds")

refunds_clean = (
    refunds
    .withColumn("refund_id", F.trim(F.col("refund_id")))
    .withColumn("return_id", F.trim(F.col("return_id")))
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn(
        "refund_requested_at",
        F.to_timestamp("refund_requested_at")
    )
    .withColumn(
        "refund_processed_at",
        F.to_timestamp("refund_processed_at")
    )
    .withColumn(
        "refund_amount",
        F.col("refund_amount").cast("double")
    )
    .withColumn(
        "refund_status",
        F.trim(F.col("refund_status"))
    )
    .withColumn(
        "refund_method",
        F.trim(F.col("refund_method"))
    )
)

# ------------------------------------------------------------
# Validate refunds against Silver returns
# ------------------------------------------------------------

refunds_with_returns = refunds_clean.join(
    returns_silver.select(
        "return_id",
        "order_id"
    ).withColumn("_return_match", F.lit(1)),
    on=["return_id", "order_id"],
    how="left"
)


# ------------------------------------------------------------
# Invalid refunds
# ------------------------------------------------------------

refunds_invalid = (
    refunds_with_returns
    .filter(
        F.col("refund_id").isNull() |
        (F.col("refund_id") == "") |
        F.col("return_id").isNull() |
        F.col("_return_match").isNull() |
        F.col("refund_requested_at").isNull() |
        F.col("refund_amount").isNull() |
        (F.col("refund_amount") < 0) |
        (
            F.col("refund_processed_at").isNotNull() &
            (
                F.col("refund_processed_at") <
                F.col("refund_requested_at")
            )
        )
    )
    .withColumn(
        "quarantine_reason",
        F.concat_ws(
            "; ",
            F.when(
                F.col("refund_id").isNull() |
                (F.col("refund_id") == ""),
                F.lit("Missing refund_id")
            ),
            F.when(
                F.col("return_id").isNull(),
                F.lit("Missing return_id")
            ),
            F.when(
                F.col("_return_match").isNull(),
                F.lit("Orphan refund - return not found")
            ),
            F.when(
                F.col("refund_requested_at").isNull(),
                F.lit("Invalid refund request timestamp")
            ),
            F.when(
                F.col("refund_amount").isNull() |
                (F.col("refund_amount") < 0),
                F.lit("Invalid refund amount")
            ),
            F.when(
                F.col("refund_processed_at").isNotNull() &
                (
                    F.col("refund_processed_at") <
                    F.col("refund_requested_at")
                ),
                F.lit("Processed before requested")
            )
        )
    )
    .drop("_return_match")
)

write_quarantine(refunds_invalid, "refunds_rejected")


# ------------------------------------------------------------
# Valid refunds
# ------------------------------------------------------------

refunds_valid = (
    refunds_with_returns
    .filter(
        F.col("refund_id").isNotNull() &
        (F.col("refund_id") != "") &
        F.col("return_id").isNotNull() &
        F.col("_return_match").isNotNull() &
        F.col("refund_requested_at").isNotNull() &
        F.col("refund_amount").isNotNull() &
        (F.col("refund_amount") >= 0) &
        (
            F.col("refund_processed_at").isNull() |
            (
                F.col("refund_processed_at") >=
                F.col("refund_requested_at")
            )
        )
    )
    .drop("_return_match")
)
# ------------------------------------------------------------
# Remove duplicate refunds
# ------------------------------------------------------------

refund_window = Window.partitionBy("refund_id").orderBy(
    F.col("refund_requested_at").asc()
)

refunds_ranked = (
    refunds_valid
    .withColumn(
        "duplicate_rank",
        F.row_number().over(refund_window)
    )
)

refunds_duplicates = (
    refunds_ranked
    .filter(F.col("duplicate_rank") > 1)
    .drop("duplicate_rank")
    .withColumn(
        "quarantine_reason",
        F.lit("Duplicate refund_id")
    )
)

write_quarantine(refunds_duplicates, "refunds_duplicates")


refunds_silver = (
    refunds_ranked
    .filter(F.col("duplicate_rank") == 1)
    .drop("duplicate_rank")
)

(
    refunds_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{SILVER}.refunds")
)

print("Created Silver table: refunds")


# ============================================================
# FINAL VALIDATION
# ============================================================

print("\n========== SILVER TABLE COUNTS ==========")

for table_name in [
    "customers",
    "sellers",
    "products",
    "orders",
    "returns",
    "refunds"
]:
    count = spark.table(
        f"{SILVER}.{table_name}"
    ).count()

    print(f"{table_name}: {count:,} rows")


print("\n========== SILVER LAYER COMPLETE ==========")